In [1]:
# notebooks/01_tax_optimization_demo.ipynb
# Run cells in order. Install deps first: pip install taxopt[notebook]

In [2]:
# Cell 1 — Imports
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import time
import plotly.graph_objects as go
import plotly.express as px
from datetime import date
from dataclasses import replace
from typing import cast

from taxopt import (
    Portfolio, USCapitalGainsPolicy, LotMethod,
    CvxpyOptimizer, PortfolioPolicy, OptimizationInputs,
    TaxLedger,
)


In [3]:
# Cell 2 — Download price data
# Largest stock
# TICKERS_1: list[str] = [
#     "NVDA", "AAPL", "MSFT", "AMZN", "GOOGL", "AVGO", "GOOG", "META", "TSLA", "BRK-B",
#     "JPM", "LLY", "XOM", "JNJ", "WMT", "V", "MU", "COST", "MA", "NFLX",
#     "ABBV", "CVX", "PLTR", "PG", "HD", "CAT", "AMD", "GE", "BAC", "CSCO",
# ]
# Dow Jones 30
TICKERS_2: list[str] = [
    "GS", "CAT", "MSFT", "AMGN", "HD", "SHW", "MCD", "AXP", "V", "JPM",
    "TRV", "UNH", "AAPL", "JNJ", "IBM", "HON", "AMZN", "BA", "CVX", "CRM",
    "NVDA", "MMM", "PG", "WMT", "MRK", "DIS", "CSCO", "KO", "VZ", "NKE",
]
# Major ETFs across asset classes
TICKERS_3: list[str] = [
    "VTI", "SPY", "QQQ", "SCHD", "IJR", "VUG", "VTV", "IVE", "IVW", "VO",
    "VEA", "VWO", "IEFA", "EEM", "VXUS", "EWJ", "EZU", "MCHI", "AGG", "BND",
    "BNDX", "SHY", "TLT", "LQD", "VNQ", "GLD", "DBC", "MLPA", "VGT", "XLV",
]
TICKERS = TICKERS_2 + TICKERS_3
n: int = len(TICKERS)

_raw = yf.download(TICKERS, start="2013-01-01", end="2026-03-31", auto_adjust=True)
if _raw is None:
    raise RuntimeError("yf.download returned None")
raw: pd.DataFrame = cast(pd.DataFrame, _raw["Close"])[TICKERS].dropna(how="all")
returns: pd.DataFrame = cast(pd.DataFrame, raw.pct_change().dropna(how="all"))
print(f"Price data: {raw.index[0].date()} → {raw.index[-1].date()}, {len(raw)} days")
raw.tail(3)


[*********************100%***********************]  60 of 60 completed


Price data: 2013-01-02 → 2026-03-30, 3330 days


Ticker,GS,CAT,MSFT,AMGN,HD,SHW,MCD,AXP,V,JPM,...,BNDX,SHY,TLT,LQD,VNQ,GLD,DBC,MLPA,VGT,XLV
Date,,,,,,,,,,,,,,,,,,,,,
2026-03-26,822.640015,703.190002,365.970001,353.160004,328.429993,319.549988,308.929993,298.446503,305.529999,290.174957,...,47.628723,81.974052,85.767311,107.438545,87.720001,400.640015,28.440001,55.459999,696.039978,145.740005
2026-03-27,802.890015,695.400024,356.769989,348.769989,321.649994,315.369995,305.899994,291.348938,295.519989,281.399872,...,47.548908,82.143539,85.299179,107.179611,87.000000,414.700012,29.100000,54.950001,681.000000,143.259995
2026-03-30,807.599976,667.429993,358.959991,349.000000,323.500000,315.899994,308.529999,296.552490,299.540009,282.325134,...,47.678608,82.253212,86.434639,107.866783,87.330002,414.579987,29.260000,54.619999,668.700012,143.820007


In [4]:
# Cell 3 — Helper functions
def get_prices(as_of: date) -> dict[str, float]:
    ts = raw.index[raw.index <= pd.Timestamp(as_of)]
    row = raw.iloc[0 if len(ts) == 0 else -1] if not len(ts) else raw.loc[ts[-1]]
    return {str(k): float(v) for k, v in row[TICKERS].items()}


def get_inputs(
    as_of: date,
    policy: PortfolioPolicy,
    lookback_days: int = 252,
) -> OptimizationInputs:
    end = pd.Timestamp(as_of)
    rets_all = returns.loc[end - pd.Timedelta(days=lookback_days * 2) : end]

    # Only assets with a FULL lookback_days * 2 window of non-NaN — no partial history
    available = [
        t for t in rets_all.columns
        if rets_all[t].notna().sum() >= lookback_days
    ]
    if not available:
        raise ValueError(f"No assets with sufficient history on {as_of}")

    rets = rets_all[available].dropna()
    win  = rets.tail(lookback_days)
    n_avail = len(available)

    cov = win.cov().to_numpy() * 252 + np.eye(n_avail) * 1e-6  # annualized covariance with small diagonal regularization

    # Momentum window: needs at least 252 rows before the last 20
    mom_rets = rets.iloc[-252:-20]
    if len(mom_rets) < 20:
        # fallback: use whatever we have
        mom_rets = rets.iloc[:-20] if len(rets) > 20 else rets

    # Alpha: 12-1 month momentum → rank → z-score → beta-neutralize → risk-normalize
    momentum = (1 + mom_rets).prod() - 1
    ranked   = momentum.rank().to_numpy(dtype=float)
    w        = (ranked - ranked.mean()) / (ranked.std() + 1e-8)  # standardized signal

    b = _compute_betas(win)
    den = (b @ b) or 1.0 
    v = w - (w @ b / den) * b  # beta-neutralize
    v /= np.sqrt(max(float(v @ cov @ v), 1e-8))  # risk-normalize to target unit variance
    alpha = {t: float((cov @ v)[i]) for i, t in enumerate(available)}

    return OptimizationInputs(
        alpha=alpha,
        covariance=cov,
        assets=available,
        prices=get_prices(as_of),
        as_of=as_of,
        risk_aversion=policy.risk_aversion,
        tax_aversion=policy.tax_aversion,
        gross_leverage=policy.gross_leverage,
        net_exposure=policy.net_exposure,
        max_weight=policy.max_weight,
        max_turnover=policy.max_turnover,
    )


def _compute_betas(win: pd.DataFrame) -> np.ndarray:
    """Beta vs equal-weight index. Accepts pre-sliced window."""
    r_m  = win.mean(axis=1).to_numpy(dtype=float)
    r_mc = r_m - r_m.mean()
    denom = float((r_mc ** 2).sum())
    if denom < 1e-10:
        return np.zeros(len(win.columns))
    return np.array([
        float(((win[t].to_numpy() - win[t].mean()) * r_mc).sum()) / denom
        for t in win.columns
    ])


In [5]:
# Cell 4 — rebalance loop with ledger
POLICY = USCapitalGainsPolicy(lot_method=LotMethod.MIN_GAIN)

# ── Portfolio policy ──────────────────────────────────────────
POLICY_OPT = PortfolioPolicy(
    risk_aversion  = 2.0,
    tax_aversion   = 1.0,
    gross_leverage = 1.6,   # L + S  (e.g. L130/S30 → 1.6)
    net_exposure   = 1.0,   # L - S
    max_weight     = 0.2,
    max_turnover   = 0.5,  # None = unlimited
)

# ── Solver settings ───────────────────────────────────────────
SOLVER = CvxpyOptimizer(
    solver="SCIP",
    verbose=False,
    tax_aware=True,                 # enable tax-aware optimization (enable objective term)
    relax_turnover=True,            # gradually widens turnover if infeasible
    turnover_relax_step=0.05,       # +5pp per attempt
    turnover_relax_max_attempts=5,  # up to +25pp before giving up
    mip_gap=0.02,                   # larger MIP gap for faster solve at the cost of optimality guarantee
)

# All month-ends in range; rebalance on each except the last,
# which serves only as the EOM date of the final holding period.
all_dates: list[date] = pd.date_range("2016-02-01", "2026-03-31", freq="ME").date.tolist()
rebal_dates = all_dates[:-1]
eom_dates   = all_dates[1:]

portfolio = Portfolio(cash=100_000.0)
ledger    = TaxLedger()

history: list[dict] = []

for i, (rebal_date, eom_date) in enumerate(zip(rebal_dates, eom_dates)):
    is_first   = (i == 0)

    # --- BOM prices & NAV ---
    prices_bom = get_prices(rebal_date)
    tv_pre     = portfolio.total_value(prices_bom)

    # --- Optimization inputs ---
    inputs = get_inputs(
        rebal_date,
        policy=replace(POLICY_OPT, max_turnover=None if is_first else POLICY_OPT.max_turnover),
    )

    # --- Solve at BOM prices ---
    t0     = time.perf_counter()
    result = SOLVER.solve(portfolio, inputs, POLICY, tv_pre)
    elapsed = time.perf_counter() - t0

    if result.status not in ("optimal", "optimal_inaccurate"):
        print(f"{rebal_date}: {result.status}, skipping")
        continue
    
    # --- Apply trades at BOM prices; this is post-optimization portfolio ---
    portfolio, tax_report = portfolio.apply_actions(result.actions, prices_bom, POLICY, rebal_date)
    tv_post = portfolio.total_value(prices_bom)
    assert abs(tv_post - tv_pre) / tv_pre < 1e-4, f"NAV not conserved: {tv_pre:.2f} → {tv_post:.2f}"

    # --- Post-optimization weights at BOM (rebal_date) ---
    weights = portfolio.weights(prices_bom)

    # --- Tax alpha this period ---
    st = tax_report.totals_by_type.get("short_term", 0.0)
    lt = tax_report.totals_by_type.get("long_term", 0.0)
    tax_alpha = -(st * POLICY.st_rate + lt * POLICY.lt_rate) / tv_pre

    # --- EOM drift ---
    prices_eom = get_prices(eom_date)
    tv_eom     = portfolio.total_value(prices_eom)

    ledger.record(tax_report)
    at_nav_eom = ledger.after_tax_nav(portfolio, prices_eom, eom_date, POLICY)

    history.append({
        "date":          rebal_date,
        "nav_eom":       tv_eom,
        "after_tax_nav": at_nav_eom,
        "st_realized":   st,
        "lt_realized":   lt,
        "st_cumulative": ledger.st_realized,
        "lt_cumulative": ledger.lt_realized,
        "tax_alpha":     tax_alpha,
        "num_actions":   len(result.actions),
        "num_lots":      sum(len(v) for v in portfolio.lots.values()),
        "solve_time":    elapsed,
        "eff_turnover":  result.effective_turnover,
        "weights":       weights,
    })

    relaxed = (inputs.max_turnover and result.effective_turnover > inputs.max_turnover + 1e-4)
    note = f" [relaxed → {result.effective_turnover:.0%}]" if relaxed else ""
    to_str = f"{result.effective_turnover:.0%}"
    print(
        f"{eom_date} EOM=${tv_eom:>10,.0f} AT-NAV=${at_nav_eom:>10,.0f} "
        f"ST={st:>+8,.0f} "
        f"LT={lt:>+8,.0f} "
        f"TaxAlpha={tax_alpha*100:>+6.3f}% "
        f"TO={to_str:>4}{note} "
        f"[{elapsed:.1f}s]"
    )

df = pd.DataFrame(history).set_index("date")


2016-03-31 EOM=$   102,644 AT-NAV=$   100,630 ST=      +0 LT=      +0 TaxAlpha=-0.000% TO= 80% [190.5s]
2016-04-30 EOM=$   104,445 AT-NAV=$   102,575 ST=  -3,089 LT=      +0 TaxAlpha=+1.053% TO= 34% [38.6s]
2016-05-31 EOM=$   110,768 AT-NAV=$   106,811 ST=    -519 LT=      +0 TaxAlpha=+0.174% TO= 39% [44.1s]
2016-06-30 EOM=$   112,825 AT-NAV=$   108,043 ST=    -259 LT=      +0 TaxAlpha=+0.082% TO= 24% [2.0s]
2016-07-31 EOM=$   117,361 AT-NAV=$   110,611 ST=    -740 LT=      +0 TaxAlpha=+0.230% TO= 39% [63.5s]
2016-08-31 EOM=$   116,717 AT-NAV=$   110,320 ST=  -1,806 LT=      +0 TaxAlpha=+0.538% TO= 35% [0.6s]
2016-09-30 EOM=$   120,788 AT-NAV=$   113,271 ST=    -553 LT=      +0 TaxAlpha=+0.166% TO= 25% [0.4s]
2016-10-31 EOM=$   115,017 AT-NAV=$   108,426 ST=    +793 LT=      +0 TaxAlpha=-0.230% TO= 34% [42.2s]
2016-11-30 EOM=$   120,407 AT-NAV=$   112,467 ST=  -2,526 LT=      +0 TaxAlpha=+0.769% TO= 50% [42.6s]
2016-12-31 EOM=$   124,815 AT-NAV=$   115,785 ST=  +1,573 LT=      +0 TaxAl

In [6]:
# Cell 5 — Summary metrics
def _row(label, val, fmt="$"):
    if   fmt == "$": print(f"  {label + ':':<32} ${val:>10,.0f}")
    elif fmt == "%": print(f"  {label + ':':<32}  {val:>10.2f}%")
    elif fmt == "f": print(f"  {label + ':':<32}  {val:>10.1f}")
    elif fmt == "s": print(f"  {label + ':':<32}  {val:>10.2f}s")
    elif fmt == "d": print(f"  {label + ':':<32}  {val:>10d}")

final_nav       = float(df["after_tax_nav"].iloc[-1])
pretax_nav      = float(df["nav_eom"].iloc[-1])
initial_nav     = float(df["nav_eom"].iloc[0])
total_st        = float(df["st_realized"].sum())
total_lt        = float(df["lt_realized"].sum())
total_realized  = total_st + total_lt
cum_tax         = ledger.cumulative_tax_value(POLICY)
pretax_return   = (pretax_nav  / initial_nav - 1) * 100
aftertax_return = (final_nav   / initial_nav - 1) * 100
tax_drag        = pretax_return - aftertax_return
tax_efficiency  = aftertax_return / pretax_return if pretax_return != 0 else float("nan")
tax_alpha_ann   = float(np.prod(df["tax_alpha"].to_numpy(dtype=float) + 1)) ** (12 / len(rebal_dates)) - 1
n_years         = len(rebal_dates) / 12
pretax_cagr     = (pretax_nav  / initial_nav) ** (1 / n_years) - 1
aftertax_cagr   = (final_nav   / initial_nav) ** (1 / n_years) - 1
pretax_rets    = df["nav_eom"].pct_change().dropna()
aftertax_rets  = df["after_tax_nav"].pct_change().dropna()
pretax_sharpe  = (pretax_rets.mean() / pretax_rets.std()) * np.sqrt(12)
aftertax_sharpe = (aftertax_rets.mean() / aftertax_rets.std()) * np.sqrt(12)

print("=" * 52)
print(f"  Backtest: {rebal_dates[0]} → {rebal_dates[-1]}  ({n_years:.1f} yrs)")
print("=" * 52)
_row("Initial NAV", initial_nav)
_row("Pre-tax final NAV", pretax_nav)
_row("After-tax final NAV", final_nav)
print()
_row("Pre-tax return", pretax_return, "%")
_row("Pre-tax CAGR", pretax_cagr * 100, "%")
_row("After-tax return", aftertax_return, "%")
_row("After-tax CAGR", aftertax_cagr * 100, "%")
_row("Pre-tax Sharpe", pretax_sharpe, "f")
_row("After-tax Sharpe", aftertax_sharpe, "f")
_row("Tax drag", tax_drag, "%")
_row("Tax efficiency ratio", tax_efficiency, "%")
_row("Annualized tax alpha", tax_alpha_ann * 100, "%")
print()
_row("Cumul. ST realized", total_st, "$")
_row("Cumul. LT realized", total_lt, "$")
_row("Cumul. net realized", total_realized, "$")
_row("Net tax impact (realized)", cum_tax, "$")
print()
_row("Effective rate on realized", cum_tax/max(abs(total_realized),1)*100, "%")
_row("Avg actions per rebalance", df['num_actions'].mean(), "x")
_row("Avg solve time:", df['solve_time'].mean(), "s")
_row("Final lot count:", int(df['num_lots'].iloc[-1]), "x")
print("=" * 52)

  Backtest: 2016-02-29 → 2026-02-28  (10.1 yrs)
  Initial NAV:                     $   102,644
  Pre-tax final NAV:               $   575,317
  After-tax final NAV:             $   491,819

  Pre-tax return:                       460.50%
  Pre-tax CAGR:                          18.64%
  After-tax return:                     379.15%
  After-tax CAGR:                        16.81%
  Pre-tax Sharpe:                          1.1
  After-tax Sharpe:                        1.1
  Tax drag:                              81.35%
  Tax efficiency ratio:                   0.82%
  Annualized tax alpha:                   0.78%

  Cumul. ST realized:              $  -122,219
  Cumul. LT realized:              $   156,785
  Cumul. net realized:             $    34,566
  Net tax impact (realized):       $   -11,420

  Effective rate on realized:           -33.04%
  Avg solve time::                        7.88s


In [7]:
# Cell 6 — Interactive charts
dates = df.index.astype(str).tolist()

# ── Chart 1: NAV vs After-Tax NAV ──────────────────────────────────────────
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=dates, y=df["nav_eom"], name="Pre-Tax NAV",
                          line=dict(color="#3498db"), mode="lines+markers",
                          hovertemplate="%{x}<br>Pre-Tax NAV: $%{y:,.0f}<extra></extra>"))
fig1.add_trace(go.Scatter(x=dates, y=df["after_tax_nav"], name="After-Tax NAV",
                          line=dict(color="#2ecc71"), mode="lines+markers",
                          hovertemplate="%{x}<br>After-Tax NAV: $%{y:,.0f}<extra></extra>"))
fig1.update_layout(title="NAV vs After-Tax NAV (EOM)", hovermode="x unified",
                   xaxis_title="Date", yaxis_title="$ Value")
fig1.show()

# ── Chart 2: Realized ST / LT Gains per Rebalance ─────────────────────────
fig2 = go.Figure()
fig2.add_trace(go.Bar(x=dates, y=df["st_realized"], name="ST Realized",
                      marker_color="#e74c3c",
                      hovertemplate="%{x}<br>ST: $%{y:,.0f}<extra></extra>"))
fig2.add_trace(go.Bar(x=dates, y=df["lt_realized"], name="LT Realized",
                      marker_color="#2ecc71",
                      hovertemplate="%{x}<br>LT: $%{y:,.0f}<extra></extra>"))
fig2.update_layout(title="Realized ST / LT Gains per Rebalance",
                   barmode="group", hovermode="x unified",
                   xaxis_title="Date", yaxis_title="$ Gain / Loss")
fig2.show()

# ── Chart 3: Cumulative Realized ST / LT ──────────────────────────────────
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=dates, y=df["st_cumulative"], name="ST Cumulative",
                          line=dict(color="#e74c3c"), mode="lines+markers",
                          hovertemplate="%{x}<br>ST Cumul: $%{y:,.0f}<extra></extra>"))
fig3.add_trace(go.Scatter(x=dates, y=df["lt_cumulative"], name="LT Cumulative",
                          line=dict(color="#2ecc71"), mode="lines+markers",
                          hovertemplate="%{x}<br>LT Cumul: $%{y:,.0f}<extra></extra>"))
fig3.update_layout(title="Cumulative Realized ST / LT Gains", hovermode="x unified",
                   xaxis_title="Date", yaxis_title="$ Cumulative")
fig3.show()

# ── Chart 4: Cumulative Tax Alpha ──────────────────────────────────────────
cum_alpha = df["tax_alpha"].astype(float).cumsum() * 100
fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=dates, y=cum_alpha, name="Cumul. Tax Alpha",
                          line=dict(color="#9b59b6"), mode="lines+markers",
                          fill="tozeroy", fillcolor="rgba(155,89,182,0.15)",
                          hovertemplate="%{x}<br>Tax Alpha: %{y:.3f}%<extra></extra>"))
fig4.add_hline(y=0, line_dash="dash", line_color="gray")
fig4.update_layout(title="Cumulative Tax Alpha (% of NAV)", hovermode="x unified",
                   xaxis_title="Date", yaxis_title="% of NAV")
fig4.show()

# ── Chart 5: Portfolio allocation over time ────────────────────────────────
weights_df = (
    pd.DataFrame(df["weights"].tolist(), index=df.index)
    .astype(float)
    .round(4)
    .fillna(0.0)
)

long_weights  = weights_df.clip(lower=0)
short_weights = weights_df.clip(upper=0)

long_cols  = [a for a in weights_df.columns if (long_weights[a]  > 0).any()]
short_cols = [a for a in weights_df.columns if (short_weights[a] < 0).any()]

all_cols  = long_cols + short_cols
colors    = px.colors.sample_colorscale("turbo", [i / max(len(all_cols) - 1, 1) for i in range(len(all_cols))])
color_map = dict(zip(all_cols, colors))

fig5 = go.Figure()

for asset in long_cols:
    fig5.add_trace(go.Scatter(
        x=dates, y=long_weights[asset], name=asset,
        mode="none", stackgroup="longs",
        fillcolor=color_map[asset],
    ))

for asset in short_cols:
    fig5.add_trace(go.Scatter(
        x=dates, y=short_weights[asset], name=f"{asset} (short)",
        mode="none", stackgroup="shorts",
        fillcolor=color_map[asset],
    ))

fig5.update_layout(
    title="Portfolio Allocation Over Time",
    xaxis_title="Date", yaxis_title="Weight",
    yaxis=dict(tickformat=".0%"),
    hovermode="x unified",
)
fig5.show()

In [8]:
# Cell 7 — Lot inspection (as of final EOM)
final_date   = eom_dates[-1]          # ← was rebal_dates[-1]
final_prices = get_prices(final_date) # prices ~1 month later

rows = [
    {
        "asset":       asset,
        "qty":         round(lot.quantity, 4),
        "basis":       round(lot.cost_basis, 2),
        "price":       round(final_prices[asset], 2),
        "unreal_gain": round((final_prices[asset] - lot.cost_basis) * lot.quantity, 2),
        "days_held":   (final_date - lot.acquisition_date).days,
        "gain_type":   POLICY.classify_gain(lot, final_date, 0.0),
    }
    for asset, lots in portfolio.lots.items()
    for lot in lots
]
lot_df = pd.DataFrame(rows).sort_values("unreal_gain")
unrealized = lot_df.groupby("gain_type")["unreal_gain"].sum()
print(f"Total unrealized: ${lot_df['unreal_gain'].sum():,.0f}")
print(f"ST unrealized:    ${unrealized.get('short_term', 0):,.0f}")
print(f"LT unrealized:    ${unrealized.get('long_term', 0):,.0f}")
lot_df

Total unrealized: $440,751
ST unrealized:    $53,680
LT unrealized:    $387,071


,asset,qty,basis,price,unreal_gain,days_held,gain_type
96,BA,62.3629,227.53,189.21,-2389.75,31,short_term
158,GLD,69.1825,444.95,414.58,-2101.07,59,short_term
21,BNDX,0.0001,41.35,47.68,0.00,1278,long_term
25,BNDX,0.0007,41.42,47.68,0.00,1186,long_term
20,BNDX,0.0001,40.28,47.68,0.00,3622,long_term
...,...,...,...,...,...,...,...
127,NKE,-519.8278,85.37,51.24,17739.54,547,short_term
34,AAPL,110.8655,33.22,246.63,23659.62,3257,long_term
8,AMZN,325.2647,27.63,200.95,56376.17,3683,long_term
12,NVDA,558.8295,0.77,165.17,91872.86,3683,long_term
